# 🚗 Vehicle Fuel Efficiency Prediction
## Notebook 6: Model Building

---

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    XGBOOST_AVAILABLE = False
    print('XGBoost not installed — skipping')
plt.style.use('seaborn-v0_8-whitegrid')
RANDOM_STATE = 42
print('Libraries loaded ✓')

Libraries loaded ✓


In [6]:
df = pd.read_csv('../data/auto-mpg-features.csv')
TARGET = 'mpg'
FEATURE_COLS = ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'power_to_weight', 'displacement_per_cyl', 'log_weight', 'log_displacement', 'log_horsepower', 'weight_class_enc', 'era_enc', 'origin_2', 'origin_3']
X = df[FEATURE_COLS]
y = df[TARGET]
print(f'Features: {X.shape[1]} | Samples: {X.shape[0]}')

Features: 15 | Samples: 398


## 6.1 Train-Test Split

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

Train: (318, 15) | Test: (80, 15)


## 6.2 Baseline Model Comparison (5-Fold CV)

In [8]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
models = {
    'Linear Regression': Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())]),
    'Ridge':             Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=1.0))]),
    'Lasso':             Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=0.1))]),
    'ElasticNet':        Pipeline([('scaler', StandardScaler()), ('model', ElasticNet(alpha=0.1))]),
    'KNN':               Pipeline([('scaler', StandardScaler()), ('model', KNeighborsRegressor(n_neighbors=7))]),
    'SVR':               Pipeline([('scaler', StandardScaler()), ('model', SVR(C=10, epsilon=0.1))]),
    'Decision Tree':     DecisionTreeRegressor(max_depth=5, random_state=RANDOM_STATE),
    'Random Forest':     RandomForestRegressor(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
    'Extra Trees':       ExtraTreesRegressor(n_estimators=200, max_depth=10, random_state=RANDOM_STATE),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=RANDOM_STATE),
}
if XGBOOST_AVAILABLE:
    models['XGBoost'] = XGBRegressor(n_estimators=200, learning_rate=0.1, max_depth=4,
                                     subsample=0.8, colsample_bytree=0.8,
                                     random_state=RANDOM_STATE, verbosity=0)
results = []
for name, model in models.items():
    cv_r2   = cross_val_score(model, X_train, y_train, cv=kf, scoring='r2')
    cv_rmse = np.sqrt(-cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_mean_squared_error'))
    cv_mae  = -cross_val_score(model, X_train, y_train, cv=kf, scoring='neg_mean_absolute_error')
    results.append({
        'Model': name,
        'CV R²': f"{cv_r2.mean():.4f} ± {cv_r2.std():.4f}",
        'CV RMSE': f"{cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}",
        'CV MAE': f"{cv_mae.mean():.4f} ± {cv_mae.std():.4f}",
        'R²_mean': cv_r2.mean(),
        'RMSE_mean': cv_rmse.mean(),
    })
    print(f'{name:<22} R²={cv_r2.mean():.4f} | RMSE={cv_rmse.mean():.4f}')
results_df = pd.DataFrame(results).sort_values('R²_mean', ascending=False)
display(results_df[['Model', 'CV R²', 'CV RMSE', 'CV MAE']])

Linear Regression      R²=0.8468 | RMSE=3.0615
Ridge                  R²=0.8479 | RMSE=3.0517
Lasso                  R²=0.8349 | RMSE=3.1844
ElasticNet             R²=0.8324 | RMSE=3.2080
KNN                    R²=0.8373 | RMSE=3.1572
SVR                    R²=0.8698 | RMSE=2.8200
Decision Tree          R²=0.7492 | RMSE=3.8790
Random Forest          R²=0.8528 | RMSE=2.9869
Extra Trees            R²=0.8513 | RMSE=2.9961
Gradient Boosting      R²=0.8568 | RMSE=2.9508
XGBoost                R²=0.8609 | RMSE=2.9189


,Model,CV R²,CV RMSE,CV MAE
5,SVR,0.8698 ± 0.0185,2.8200 ± 0.1891,1.9373 ± 0.1313
10,XGBoost,0.8609 ± 0.0170,2.9189 ± 0.2057,2.0240 ± 0.1745
9,Gradient Boosting,0.8568 ± 0.0238,2.9508 ± 0.1744,2.0617 ± 0.1368
7,Random Forest,0.8528 ± 0.0256,2.9869 ± 0.0974,2.0328 ± 0.0782
8,Extra Trees,0.8513 ± 0.0303,2.9961 ± 0.1543,1.9825 ± 0.1148
1,Ridge,0.8479 ± 0.0173,3.0517 ± 0.1653,2.2502 ± 0.1849
0,Linear Regression,0.8468 ± 0.0189,3.0615 ± 0.1788,2.2553 ± 0.1885
4,KNN,0.8373 ± 0.0184,3.1572 ± 0.2008,2.2487 ± 0.1998
2,Lasso,0.8349 ± 0.0141,3.1844 ± 0.1813,2.3669 ± 0.1780
3,ElasticNet,0.8324 ± 0.0128,3.2080 ± 0.1650,2.3735 ± 0.1748


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plot_df = results_df.sort_values('R²_mean', ascending=True)
colors_r2 = ['#2ecc71' if r > 0.90 else '#3498db' if r > 0.85 else '#e74c3c' for r in plot_df['R²_mean']]
axes[0].barh(plot_df['Model'], plot_df['R²_mean'], color=colors_r2, edgecolor='white')
axes[0].axvline(0.90, color='green', linestyle='--', linewidth=1.5, label='Target R² = 0.90')
axes[0].set_title('CV R² Score by Model', fontsize=13, fontweight='bold')
axes[0].set_xlabel('R² Score')
axes[0].legend()
colors_rmse = ['#2ecc71' if r < 3 else '#e74c3c' for r in plot_df['RMSE_mean']]
axes[1].barh(plot_df['Model'], plot_df['RMSE_mean'], color=colors_rmse, edgecolor='white')
axes[1].axvline(3.0, color='red', linestyle='--', linewidth=1.5, label='Target RMSE < 3')
axes[1].set_title('CV RMSE by Model', fontsize=13, fontweight='bold')
axes[1].set_xlabel('RMSE')
axes[1].legend()
plt.tight_layout()
plt.savefig('../plots/10_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1600x600 with 2 Axes>

## 6.3 Hyperparameter Tuning — Top Models

In [10]:
from sklearn.model_selection import GridSearchCV
rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [6, 8, 10, None],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2'],
}
rf_grid = GridSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE),
    rf_params, cv=5, scoring='r2', n_jobs=-1, verbose=0
)
rf_grid.fit(X_train, y_train)
print(f'Best RF params: {rf_grid.best_params_}')
print(f'Best RF CV R²: {rf_grid.best_score_:.4f}')

Best RF params: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_split': 2, 'n_estimators': 300}
Best RF CV R²: 0.8508


In [11]:
gb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [3, 4, 5],
    'subsample': [0.7, 0.8, 1.0],
}
gb_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=RANDOM_STATE),
    gb_params, cv=5, scoring='r2', n_jobs=-1, verbose=0
)
gb_grid.fit(X_train, y_train)
print(f'Best GB params: {gb_grid.best_params_}')
print(f'Best GB CV R²: {gb_grid.best_score_:.4f}')

Best GB params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Best GB CV R²: 0.8584


In [12]:
best_rf = rf_grid.best_estimator_
best_gb = gb_grid.best_estimator_
joblib.dump(best_rf, '../models/random_forest_best.pkl')
joblib.dump(best_gb, '../models/gradient_boosting_best.pkl')
print('Models saved ✓')

Models saved ✓
